# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This contract serves my lane — **Lane 4, CTR / Engagement Opportunity Scoring**, framed in ML-02/03:
a ranked review queue of pages that under-capture the CTR their search position deserves,
judged by Precision@50. Here that framing moves from the starter CSV onto the real warehouse
release, gets written down as a contract, and every claim is proven with a query.

## 0. Setup — connect to the hosted release

One-time (2 minutes): request access on the dataset page [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse) (instant approval), then create a **plain Read** token at HF Settings → Access Tokens.

The token is resolved env var → Colab Secret `HF_TOKEN` → hidden prompt, and handed straight to
DuckDB — it is never pasted into a cell, because this repo is public.

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
# explicit partition paths — hf:// supports no {brace} globs
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

In [ ]:
# Session sanity check (metadata-cheap): am I pointed at the right partitions?
for name, src in [('fact_daily month=2026-03', MAR), ('fact_daily month=2026-04', APR)]:
    n, d1, d2 = con.sql(f'SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {src}').fetchone()
    print(f'{name}: {n:,} rows, {d1} .. {d2}')

## 1. Unit of analysis + time window

**One row = one content page × one client, summarized over March 2026.**
The warehouse row is one *page-day*; my lane reviews *pages*, so the first thing the contract
does is fix the roll-up: every day of a month collapses into one row per page before anything else happens.

The five plain-words answers:

1. **What one row means:** one pseudonymized content page belonging to one client, aggregating all 31 days of March 2026 (summed impressions and clicks, averaged position).
2. **Which tables:** `fact_content_daily_performance` — only the `month=2026-03` partition for features; `month=2026-04` is read solely to build the label. `dim_content` would supply read-only context (e.g. `content_type`) joined with `ANY_VALUE` semantics — never summed.
3. **Time window:** features cover 2026-03-01 → 2026-03-31 (everything knowable the morning of April 1); the label is measured strictly after, 2026-04-01 → 2026-04-30. June 2026 (`_sample`) is a sealed test month and is never touched.
4. **What I predict / rank:** a binary label — *"this page under-captured its position tier in April"* — defined as April CTR < 0.5 × the April tier-median CTR (benchmark from pages with ≥1,000 April impressions; the ML-03 proxy moved into the future month). Output is a ranked review queue, scored by Precision@50.
5. **Deliberately exclude:** `fact_content_query_90d` — its fixed 90-day window overlaps my outcome month, so its columns are not cleanly knowable at the decision moment; and GA4 columns, which are zero-filled before a client's `ga4_data_start` (those zeros are fake absence, not low engagement).

In [ ]:
import pandas as pd

feat_win = pd.date_range('2026-03-01', '2026-03-31', freq='D')
label_win = pd.date_range('2026-04-01', '2026-04-30', freq='D')

print(f'Feature window : {feat_win[0].date()} .. {feat_win[-1].date()}  ({len(feat_win)} days)')
print(f'Label window   : {label_win[0].date()} .. {label_win[-1].date()}  ({len(label_win)} days)')
print(f'Decision moment: 2026-04-01 — every feature above must exist before it, the label strictly after')

## 2. Fields: feature / label / context / excluded

Every field I touch sits in exactly one bucket:

| Field(s) | Bucket | Why |
|---|---|---|
| March `SUM(gsc_impressions)`, `SUM(gsc_clicks)`, `AVG(gsc_avg_position)` | **Feature** | Measured strictly before the 2026-04-01 decision moment; GSC data, not zero-filled. |
| `tier_expected_ctr` (median March CTR of same-position-tier pages with ≥1,000 impressions) | **Feature** | Benchmark estimated from the March pool alone — no April information leaks in. |
| `ctr_mar`, `pos_mar`, `log10_imp_mar`, `imp_share_client` | **Feature** | All recomputable from March data on April 1 (one line per feature in section 3). |
| April impressions/clicks → `apr_ctr`, `tier_expected_apr`, `under_captured_apr` | **Label** | The thing predicted; computed entirely from April, never available as input. |
| `client_hash_id`, `content_hash_id` | **Context** | Pseudonym IDs for grouping, joining and the client-grouped split — never model inputs. |
| `content_type` (via `dim_content`) | **Context** | Read-only descriptor; not among my five features this week. |
| Any trend/velocity column computed across the month boundary (the `trend_pct` family) | **Excluded** | Derived from the same CTR/impression signal as the label — leakage by construction. |
| GA4 engagement columns | **Excluded** | Zero-filled before `ga4_data_start`; zeros would inject fake "no engagement" signal. |
| Everything in `fact_content_query_90d` | **Excluded** | Its 90-day window overlaps the outcome month (see §1). |

In [ ]:
FEATURES = ['ctr_mar', 'pos_mar', 'log10_imp_mar', 'tier_expected_ctr', 'imp_share_client']
CONTEXT   = ['client_hash_id', 'content_hash_id']
LABEL     = ['under_captured_apr']

print('Contract in code form:')
print(f'  features ({len(FEATURES)}): {FEATURES}')
print(f'  context  : {CONTEXT}')
print(f'  label    : {LABEL}')

## 3. Prove three facts with three queries

All three run on the mid-panel month `month=2026-03`. June 2026 is never queried.

### Fact 1 — the grain: one row really is what I said

Two probes: the raw March slice must be unique at *(report_date, client, content)* — one row per
page-day; and after the monthly roll-up there must be exactly one row per *(client, content)*.
Zero rows back from either `HAVING` probe means the grain holds.

In [ ]:
q_raw_dupes = f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
"""
q_month_dupes = f"""
    WITH monthly AS (
        SELECT client_hash_id, content_hash_id
        FROM {MAR}
        GROUP BY 1, 2
    )
    SELECT client_hash_id, content_hash_id, COUNT(*) AS c
    FROM monthly
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
    LIMIT 5
"""
raw_dupes = con.sql(q_raw_dupes).df()
mon_dupes = con.sql(q_month_dupes).df()
print(f'duplicate (page-day) keys in raw March slice : {len(raw_dupes)}  <- 0 = page-day grain holds')
print(f'duplicate (page-month) keys after roll-up    : {len(mon_dupes)}  <- 0 = page-month grain holds')
if len(raw_dupes): display(raw_dupes)
if len(mon_dupes): display(mon_dupes)

### Fact 2 — my slice: how many rows, which dates

My lane's candidate pool: pages with **≥ 100 March impressions** and a **positive average
position** (visible enough to act on; `gsc_avg_position = 0` means "no data", not rank zero).

In [ ]:
q_slice = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp,
               AVG(gsc_avg_position) AS pos,
               MIN(report_date) AS d1, MAX(report_date) AS d2
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    )
    SELECT COUNT(*) AS pool_rows, MIN(d1) AS span_start, MAX(d2) AS span_end
    FROM pagemo
"""
n_pool, s1, s2 = con.sql(q_slice).fetchone()
n_daily, dd1, dd2 = con.sql(f'SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {MAR}').fetchone()
print(f'Raw March partition          : {n_daily:>12,} page-day rows, {dd1} .. {dd2}')
print(f'My slice (visible pages)     : {n_pool:>12,} page-month rows, {s1} .. {s2}')

### Fact 3 — availability: who survives the `IS TRUE` filter

Rows written before a client's `ga4_data_start` carry GA4 columns **zero-filled** with
`ga4_data_available = FALSE`. Filtering on the flag shows how much of the panel genuinely has
analytics behind it — the rest is GSC-only history, not "zero engagement".

In [ ]:
q_avail = f"""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows
    FROM {MAR}
"""
total, avail = con.sql(q_avail).fetchone()
share = avail / total if total else float('nan')
print(f'March page-day rows total                    : {total:>12,}')
print(f'rows surviving ga4_data_available IS TRUE    : {avail:>12,}  ({share:.1%})')
print(f'GSC-only rows (flag false)                   : {total - avail:>12,}')

### Five features, max

Built from the same March partition; the label joins from April. One row = one page-month
(the §1 contract), one line per feature saying *when it becomes knowable*:

| # | Feature | Knowable at the decision moment (2026-04-01) because… |
|---|---|---|
| 1 | `ctr_mar` — March clicks ÷ March impressions | it summarizes only March 1–31 GSC activity, which closed the night before. |
| 2 | `pos_mar` — March average position | it averages March page-days only; nothing from April enters. |
| 3 | `log10_imp_mar` — log₁₀ of March impressions | it is a transform of a March total, fixed once March ends. |
| 4 | `tier_expected_ctr` — median March CTR among same-tier pages with ≥1,000 impressions | the benchmark is estimated from the March pool alone, no peeking forward. |
| 5 | `imp_share_client` — the page's share of its client's March impressions | both numerator and denominator are March sums within the same month. |

In [ ]:
import numpy as np

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS imp_mar,
               SUM(gsc_clicks)      AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
mar = con.sql(q_mar).df()

q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

frame = mar.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()

def tier_of(pos):
    if pos <= 3:   return 'p1_top'
    if pos <= 10:  return 'p1'
    if pos <= 20:  return 'p2'
    return 'deep'

frame['ctr'] = frame['clk_mar'] / frame['imp_mar']
frame['tier'] = frame['pos_mar'].apply(tier_of)

bench = (frame[frame['imp_mar'] >= 1000]
         .groupby('tier')['ctr'].median().rename('tier_expected_ctr'))
frame['tier_expected_ctr'] = frame['tier'].map(bench)
frame['ctr_mar'] = frame['ctr']
frame['log10_imp_mar'] = np.log10(frame['imp_mar'])
frame['imp_share_client'] = frame['imp_mar'] / frame['cli_imp']

# label side: April benchmark, same construction, strictly after March
frame['apr_ctr'] = frame['clk_apr'] / frame['imp_apr']
bench_apr = (frame[frame['imp_apr'] >= 1000]
             .groupby('tier')['apr_ctr'].median().rename('tier_expected_apr'))
frame['tier_expected_apr'] = frame['tier'].map(bench_apr)

labeled = frame[
    (frame['imp_apr'] >= 100) &
    (frame['tier_expected_ctr'] > 0) &
    (frame['tier_expected_apr'] > 0)
].copy()
labeled['apr_gap_ratio'] = labeled['apr_ctr'] / labeled['tier_expected_apr']
labeled['under_captured_apr'] = (labeled['apr_gap_ratio'] < 0.5).astype(int)

print(f'feature frame: {len(labeled):,} page-month rows (labeled subset of {len(frame):,} merged)')
print(f'label base rate: {labeled["under_captured_apr"].mean():.3f}')
labeled[CONTEXT + FEATURES + ['apr_gap_ratio', LABEL[0]]].head()

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42

def precision_at_k(frame, cols, k=50):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    tr_idx, te_idx = next(gss.split(frame, groups=frame['client_hash_id']))
    tr, te = frame.iloc[tr_idx], frame.iloc[te_idx]
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=2000, random_state=SEED))
    model.fit(tr[cols], tr[LABEL[0]])
    proba = model.predict_proba(te[cols])[:, 1]
    k_eff = min(k, len(te))
    top = te.iloc[np.argsort(-proba)[:k_eff]]
    return {
        'P@%d' % k_eff: top[LABEL[0]].mean(),
        'base_rate_test': te[LABEL[0]].mean(),
        'n_train': len(tr), 'n_test': len(te),
        'te_index': te.index, 'proba': proba,
    }

honest = precision_at_k(labeled, FEATURES)
print('HONEST score (five March features only)')
for key in ('P@50', 'base_rate_test'):
    print(f'  {key:16}: {honest[key]:.3f}')
print(f'  train/test sizes : {honest["n_train"]:,} / {honest["n_test"]:,} (grouped by client)')

### The trap — one label-derived column, on purpose

Everyone falls into this once: a column that *describes the answer* sneaks into the features.
Here I add `apr_gap_ratio` — the label's own numerator (April CTR ÷ April tier benchmark).
It is knowable only *after* April ends, so it cannot legally sit beside March features.
Watch the score jump toward perfect, then delete the column and keep the honest number.

In [ ]:
leaky_cols = FEATURES + ['apr_gap_ratio']

leaky = precision_at_k(labeled, leaky_cols)
print('LEAKED score (same five March features + apr_gap_ratio)')
for key in ('P@50', 'base_rate_test'):
    print(f'  {key:16}: {leaky[key]:.3f}')

print()
print('Side by side (same split, same seed):')
print(f'  leaked  P@50 = {leaky["P@50"]:.3f}   <- reads the answer, worthless in production')
print(f'  honest  P@50 = {honest["P@50"]:.3f}   <- the number I keep')

del labeled['apr_gap_ratio']
assert 'apr_gap_ratio' not in labeled.columns, 'leaky column must not survive'
print()
print('apr_gap_ratio deleted — the frame carries only legal, decision-time-knowable columns.')

## 4. Data limits

**Named limitation — survivor selection.** The labeled frame keeps only pages that still earn
≥100 April impressions, so the label's population depends on the outcome window itself: a page
that vanished from search in April can never be scored, and the model trains on March survivors.
That is a choice (a label needs an observable outcome), not an accident — but it belongs in
every claim made about Precision@50.

Secondary limits, stated plainly:

- **Unbalanced panel.** Clients start GSC/GA4 history at different dates; a global calendar
  window flatters long-history clients and starves new ones. Per-client windows would be the
  fairer design (deferred to the modeling weeks).
- **Benchmark sparsity.** `tier_expected_ctr` rests on same-month medians; rare position tiers
  with few high-volume pages give noisy benchmarks, and `deep`-tier pages with near-zero
  expected CTR are dropped rather than scored against a meaningless bar.
- **GSC-only early rows.** Before `ga4_data_start`, analytics columns are zero-filled; this
  contract excludes them entirely, so engagement-side questions are simply out of scope here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.